# Assignment 4 — Pedestrian-Crossing Safety at a College Gate
**Self-contained Google Colab notebook**

This notebook creates its own small synthetic crossing scene and short frame sequence. No external dataset, image download, API key or pretrained-weight download is required. It demonstrates image processing, background subtraction, optical flow, stereo-depth arithmetic, feature classification and an integrated alarm rule.

**Academic integrity note:** The generated scene is synthetic. Outputs are demonstration results, not measurements from a real college CCTV installation. Replace the synthetic input with faculty-approved footage and manually annotated ground truth before reporting real-world performance.

In [ ]:
# 1. Install/import required libraries (OpenCV and scikit-learn are normally available in Colab)
import cv2, numpy as np, matplotlib.pyplot as plt, pandas as pd, math, time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

SEED = 42
rng = np.random.default_rng(SEED)
print("OpenCV:", cv2.__version__)
print("NumPy:", np.__version__)

## 2. Generate built-in synthetic images and a video sequence

In [ ]:
W, H = 640, 360
def make_frame(t):
    frame = np.zeros((H,W,3), dtype=np.uint8)
    frame[:] = (75, 105, 80)                         # trees/background
    cv2.rectangle(frame,(0,205),(W,H),(65,65,65),-1) # road
    for x in range(10,W,90):
        cv2.rectangle(frame,(x,278),(x+45,284),(230,230,230),-1)
    # zebra crossing
    for x in range(245,365,22):
        cv2.rectangle(frame,(x,215),(x+11,285),(235,235,235),-1)
    # moving pedestrian
    px = int(205 + 2.0*t)
    cv2.circle(frame,(px,185),9,(40,40,220),-1)
    cv2.line(frame,(px,194),(px,225),(40,40,220),5)
    cv2.line(frame,(px,205),(px-9,218),(40,40,220),3)
    cv2.line(frame,(px,205),(px+9,218),(40,40,220),3)
    # moving car
    cx = int(440 - 3.0*t)
    cv2.rectangle(frame,(cx,245),(cx+65,272),(20,80,210),-1)
    cv2.circle(frame,(cx+13,274),7,(10,10,10),-1)
    cv2.circle(frame,(cx+52,274),7,(10,10,10),-1)
    cv2.putText(frame,f"synthetic frame {t}",(12,28),cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
    return frame

frames=[make_frame(t) for t in range(30)]
plt.figure(figsize=(12,4))
plt.imshow(cv2.cvtColor(frames[10],cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title("Built-in synthetic crossing frame")
plt.show()
print("Generated",len(frames),"frames at a demonstration rate of 25 fps (1.2 seconds).")

## 3. Task 1 — Viola–Jones concepts and integral-image calculation

In [ ]:
# Demonstrate the four-corner integral-image rectangle sum
gray = cv2.cvtColor(frames[10], cv2.COLOR_BGR2GRAY)
ii = cv2.integral(gray)
x1,y1,x2,y2 = 100,100,180,160
rect_sum = int(ii[y2,x2] - ii[y1,x2] - ii[y2,x1] + ii[y1,x1])
direct_sum = int(gray[y1:y2,x1:x2].sum())
print("Integral-image rectangle sum:", rect_sum)
print("Direct pixel sum:", direct_sum)
print("Equal:", rect_sum == direct_sum)

# Haar cascades can be tested if OpenCV's built-in cascade file is present.
cascade_path = cv2.data.haarcascades + "haarcascade_fullbody.xml"
cascade = cv2.CascadeClassifier(cascade_path)
print("Full-body cascade loaded:", not cascade.empty())
if not cascade.empty():
    boxes = cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(24,48))
    print("Illustrative cascade boxes:", len(boxes))
else:
    print("Cascade file unavailable; skip detector execution. Conceptual comparison remains in the report.")

## 4. Task 2 — Detector evaluation template and metrics

In [ ]:
# Generic IoU and precision/recall helper for manually supplied boxes.
def box_iou(a,b):
    xA=max(a[0],b[0]); yA=max(a[1],b[1]); xB=min(a[2],b[2]); yB=min(a[3],b[3])
    inter=max(0,xB-xA)*max(0,yB-yA)
    areaA=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    areaB=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    return inter/(areaA+areaB-inter+1e-9)

def detection_counts(pred_boxes, gt_boxes, threshold=0.5):
    used=set(); tp=0
    for p in pred_boxes:
        candidates=[(box_iou(p,g),j) for j,g in enumerate(gt_boxes) if j not in used]
        if candidates:
            score,j=max(candidates)
            if score>=threshold: tp+=1; used.add(j)
    fp=len(pred_boxes)-tp; fn=len(gt_boxes)-tp
    precision=tp/(tp+fp+1e-9); recall=tp/(tp+fn+1e-9)
    return {"TP":tp,"FP":fp,"FN":fn,"precision":precision,"recall":recall}

# Example only: the coordinates below are demonstration annotations.
ground_truth=[(195,174,218,230),(425,240,510,282)]
predictions=[(197,174,220,230),(423,240,511,283),(20,40,45,75)]
print(detection_counts(predictions,ground_truth))

## 5. Task 3 — Background modelling: frame difference, running average and MOG2

In [ ]:
gray_frames=[cv2.cvtColor(f,cv2.COLOR_BGR2GRAY) for f in frames]
prev=gray_frames[0]
diff_masks=[]
for cur in gray_frames[1:]:
    diff=cv2.absdiff(cur,prev)
    _,mask=cv2.threshold(diff,22,255,cv2.THRESH_BINARY)
    diff_masks.append(mask)
    prev=cur

mog=cv2.createBackgroundSubtractorMOG2(history=100,varThreshold=25,detectShadows=True)
mog_masks=[mog.apply(f,learningRate=0.03) for f in frames]
running_bg=gray_frames[0].astype(np.float32)
running_masks=[]
for cur in gray_frames[1:]:
    delta=cv2.absdiff(cur,cv2.convertScaleAbs(running_bg))
    _,m=cv2.threshold(delta,22,255,cv2.THRESH_BINARY)
    running_masks.append(m)
    cv2.accumulateWeighted(cur,running_bg,0.03)

fig,axs=plt.subplots(1,3,figsize=(13,4))
axs[0].imshow(diff_masks[-1],cmap="gray"); axs[0].set_title("Frame difference")
axs[1].imshow(running_masks[-1],cmap="gray"); axs[1].set_title("Running average")
axs[2].imshow(mog_masks[-1],cmap="gray"); axs[2].set_title("MOG2 / GMM")
for a in axs:a.axis("off")
plt.tight_layout(); plt.show()
print("Foreground pixel counts:", int(np.count_nonzero(diff_masks[-1])), int(np.count_nonzero(running_masks[-1])), int(np.count_nonzero(mog_masks[-1])))

### Task 3 hand calculation

In [ ]:
# GMM pixel example from the assignment
weights=np.array([0.60,0.30,0.10],float)
means=np.array([120.,60.,200.])
sigmas=np.array([4.,6.,10.])
alpha=0.05
def gmm_update(x,weights,means,sigmas):
    match=None
    for i,(mu,sig) in enumerate(zip(means,sigmas)):
        if abs(x-mu)<2.5*sig:
            match=i; break
    new_w=(1-alpha)*weights
    if match is not None: new_w[match]+=alpha
    new_w=new_w/new_w.sum()
    if match is not None:
        rho=alpha
        old_mu=means[match]
        means[match]=old_mu+rho*(x-old_mu)
        var=(1-rho)*(sigmas[match]**2)+rho*((x-old_mu)**2)
        sigmas[match]=math.sqrt(var)
    return match,new_w,means,sigmas
match1,weights,means,sigmas=gmm_update(118,weights,means,sigmas)
print("x=118 matched component:",match1+1 if match1 is not None else None)
print("After x=118 weights/means/sigmas:",weights,means,sigmas)
match2,weights,means,sigmas=gmm_update(200,weights,means,sigmas)
print("x=200 matched component:",match2+1 if match2 is not None else None)
print("After x=200 weights/means/sigmas:",weights,means,sigmas)

## 6. Task 4 — Optical flow and speed conversion

In [ ]:
# Sparse Lucas-Kanade optical flow between two synthetic frames
old=cv2.cvtColor(frames[5],cv2.COLOR_BGR2GRAY)
new=cv2.cvtColor(frames[6],cv2.COLOR_BGR2GRAY)
p0=cv2.goodFeaturesToTrack(old,maxCorners=100,qualityLevel=0.01,minDistance=7,blockSize=7)
if p0 is not None:
    p1,st,err=cv2.calcOpticalFlowPyrLK(old,new,p0,None,winSize=(21,21),maxLevel=3,
                                       criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,30,0.01))
    good_old=p0[st.flatten()==1]; good_new=p1[st.flatten()==1]
    print("Tracked feature points:",len(good_new))
    if len(good_new):
        print("Mean image displacement (pixels/frame):",np.linalg.norm(good_new-good_old,axis=1).mean())
else:
    print("No corners detected in this synthetic frame.")

fps=25
px_per_frame=9
metres_per_pixel=5/150
speed_ms=px_per_frame*fps*metres_per_pixel
print(f"Hand calculation: {speed_ms:.2f} m/s = {speed_ms*3.6:.2f} km/h")
print("At +/-1 px/frame:", (px_per_frame-1)*fps*metres_per_pixel*3.6,
      "to", (px_per_frame+1)*fps*metres_per_pixel*3.6, "km/h")

## 7. Task 5 — Stereo depth and monocular limitations

In [ ]:
f=700.0; B=0.12
disparities=[14,13,3,2]
depths=[f*B/d for d in disparities]
print(pd.DataFrame({"disparity_px":disparities,"depth_m":depths}).to_string(index=False))
required_baseline=(20**2*0.5)/(700*0.5)
print("Baseline for <=0.5 m depth error at 20 m:",required_baseline,"m")

# Synthetic rectified pair: a horizontal shift creates a known disparity.
left=np.zeros((120,240),np.uint8)
cv2.rectangle(left,(80,35),(145,90),180,-1)
right=np.zeros_like(left)
right[:,3:]=left[:,:-3] # disparity of 3 pixels for the rectangle
plt.figure(figsize=(10,3))
plt.subplot(1,2,1); plt.imshow(left,cmap="gray"); plt.title("Left synthetic view"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(right,cmap="gray"); plt.title("Right view (3 px shift)"); plt.axis("off")
plt.show()
print("For this toy pair, nominal disparity=3 px -> depth=28 m under the assignment calibration.")

## 8. Task 6 — Pattern classifiers on generated blob features

In [ ]:
# Generate a deterministic illustrative feature dataset:
# area, aspect ratio, height, solidity, speed; labels 0=pedestrian, 1=two-wheeler, 2=four-wheeler.
n=240
centres=np.array([[420,0.42,58,0.82,1.1],[900,1.25,42,0.88,5.0],[2600,2.0,55,0.93,7.5]])
X=[]; y=[]
for cls,c in enumerate(centres):
    samples=rng.normal(c,[55,0.12,7,0.05,1.0],size=(n//3,5))
    X.append(samples); y.extend([cls]*(n//3))
X=np.vstack(X); y=np.array(y)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEED,stratify=y)
models={
 "k-NN":make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=5)),
 "SVM":make_pipeline(StandardScaler(),SVC(kernel="rbf",C=2)),
 "Decision tree":DecisionTreeClassifier(max_depth=5,random_state=SEED)
}
for name,model in models.items():
    t0=time.perf_counter(); model.fit(Xtr,ytr); train_ms=(time.perf_counter()-t0)*1000
    t0=time.perf_counter(); pred=model.predict(Xte); pred_ms=(time.perf_counter()-t0)*1000
    print("\\n",name,"accuracy:",round(accuracy_score(yte,pred),3),
          "train ms:",round(train_ms,2),"prediction ms:",round(pred_ms,2))
    print(confusion_matrix(yte,pred))
    print(classification_report(yte,pred,target_names=["pedestrian","two-wheeler","four-wheeler"],zero_division=0))

## 9. Task 7 — Still-image motion transfer demonstration

In [ ]:
# Translation-only animation from the built-in still frame.
still=frames[0].copy()
translated=[]
for k in range(8):
    M=np.float32([[1,0,2*k],[0,1,0]])
    translated.append(cv2.warpAffine(still,M,(W,H)))
fig,axs=plt.subplots(1,3,figsize=(12,3))
for ax,idx in zip(axs,[0,3,7]):
    ax.imshow(cv2.cvtColor(translated[idx],cv2.COLOR_BGR2RGB)); ax.set_title(f"Frame {idx}"); ax.axis("off")
plt.tight_layout(); plt.show()
print("Created",len(translated),"translation-only demonstration frames.")
print("A production implementation should use per-object masks, local affine transforms, occlusion masks and explicit synthetic labels.")

## 10. Task 8 — Integrated pipeline and ablation template

In [ ]:
# Example rule combining detector-like pedestrian box, estimated speed and distance.
def safety_alarm(pedestrian_present, vehicle_speed_kmh, distance_m, consecutive_frames,
                 speed_limit=30, distance_limit=15, required_frames=3):
    return bool(pedestrian_present and vehicle_speed_kmh>speed_limit
                and distance_m<distance_limit and consecutive_frames>=required_frames)

cases=[
    (True,35,12,3),
    (True,28,10,5),
    (True,40,18,4),
    (False,45,8,5)
]
for c in cases:
    print(c,"=> ALARM" if safety_alarm(*c) else "=> no alarm")

# Ablation table is a fill-in template for measurements from the actual annotated clip.
ablation=pd.DataFrame([
 ["Full hybrid pipeline","Detection + GMM + flow + depth","Record baseline"],
 ["Without GMM gate","Detector + flow + depth","Record change"],
 ["Without detector","GMM + flow + depth","Record change"],
 ["Without optical flow","Detection + GMM + depth","Record change"],
 ["Without depth","Detection + GMM + flow","Record change"]
],columns=["Configuration","Stages","Evidence to record"])
display(ablation)

## 11. Task 9 — latency budget and redesign

In [ ]:
# Illustrative engineering budget only; replace each value with measurements on the target processor.
budget=pd.DataFrame([
 ["Compact detector (INT8, resized input)",55],
 ["GMM foreground update",8],
 ["Optical-flow tracking",18],
 ["Ground-plane distance estimate",4],
 ["Tracking + alarm logic",5],
 ["Total",90]
],columns=["Stage","Illustrative milliseconds"])
display(budget)
print("Per-frame budget at 25 fps = 40 ms. A 90 ms pipeline cannot process every frame serially at 25 fps.")
print("Use asynchronous scheduling, detector every k frames, tracked boxes between detections, ROI cropping, and alarm confirmation across frames.")
print("The end-to-end warning target is 200 ms; measure capture, inference, tracking, decision and output delays separately.")

## 12. Export the generated built-in images (optional)
The images already exist in notebook memory. This cell saves sample frames into the Colab working directory.

In [ ]:
for i in [0,10,20,29]:
    cv2.imwrite(f"synthetic_crossing_{i:02d}.png",frames[i])
print("Saved four synthetic frames in the current Colab directory.")